# Minimiser le nombre de camions pour charger des palettes
 - Correction
Une entreprise de logistique dispose de $m$ camions $\{1, \dots, m\}$ afin de servir chaque jour un ensemble de clients. Dans les camions, $n$ palettes $\{1, \dots, n\}$ doivent être chargées, la $j^{ième}$ étant caractérisée par un poids $w_j$. Chaque camion a une capacité en poids de $b$. Le problème consiste à déterminer
le nombre minimal de camions pour charger toutes les palettes.

## Question 1

Écrivez un modèle de PLNE pour minimiser le nombre total de camions nécessaires pour servir tous les clients. Identifiez les décisions qui doivent être prises ainsi que les variables de décision correspondantes qui doivent être utilisées. Identifiez et commentez la fonction objectif du problème et les contraintes.


**CORRECTION :**


Soit $I = \{1, \dots, m\}$ et $J = \{1, \dots, n\}$

Pour $i \in I$, soit $y_i$ égal à 1 si le camion numéro $i$ est utilisé, et 0 sinon.

Pour $i\in I$ et $j\in J$, soit $x_{ij}$ égal à 1 si la palette numéro $j$ est transportée par le véhicule numéro $i$.

Le problème peut être formulé comme suit.

\begin{align}
&\min \sum_{i \in I} y_i\\
&\sum_{i \in I} x_{ij} = 1 & \quad \forall j \in J\\
&\sum_{j \in J} w_j~x_{ij} \le b~y_i & \quad \forall i \in I\\
&y_i \in \{0,1\} & \forall i \in I\\
&x_{ij} \in \{0,1\} & \forall i \in I, j \in J
\end{align}


## Question 2

Considérez le cas suivant dans lequel il y a 10 palettes ($n = 10$) et 10 camions disponibles ($m = 10$), la capacité de chaque camion étant de $100$. Les poids des palettes sont :

| &nbsp; | $j=1$ | $j=2$ | $j=3$ | $j=4$ | $j=5$ | $j=6$ | $j=7$ | $j=8$ | $j=9$ | $j=10$ |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| $w_1$ | 52 | 51 | 28 | 27 | 26 | 25 | 23 | 23 | 23 | 22 |

Définissez une heuristique pour trouver une bonne solution. De combien de camions avons-nous besoin ?

**CORRECTION :**


On peut classer les palettes par poids décroissant.
Pour chaque palette dans cet ordre, placez-la dans le camion ayant la capacité disponible minimale supérieure au poids de la palette, ou utilisez un nouveau camion pour la charger s'il n'existe pas un tel camion.

Lorsque l'on applique cette heuristique à l'instance, on obtient :

- Camion 1 : 52, 28
- Camion 2 : 51, 27, 22
- Camion 3 : 26, 25, 23, 23
- Camion 4 : 23

Nous avons donc besoin d'au plus 4 camions pour charger toutes les palettes.

## Question 3

Implémentez le PLNE pour résoudre à l'optimalité l'instance de la Question 2 avec $m$ égal au nombre de camions utilisés dans l'heuristique.

In [ ]:
%pip install mip

In [ ]:
pallet_weights = [ 52, 51, 28, 27, 26, 25, 23, 23, 23, 22]
truck_capacity = 100

In [ ]:
from mip import *

In [ ]:
##################
#   Correction   #
##################


def solve_packing(truck_capacity, weights, number_of_trucks):
    m = Model()
    I = range(number_of_trucks)
    J = range(len(weights))
    x = [
            [m.add_var(var_type = BINARY) for j in J] for i in I
    ]
    y = [ m.add_var(var_type = BINARY) for i in I]
    m.objective = minimize(xsum(y[i] for i in I))

    # for each pallet
    for j in J:
        m += xsum(x[i][j] for i in I) == 1

    # for each truck
    for i in I:
        m += xsum( weights[j] * x[i][j] for j in J) <= truck_capacity * y[i]
    m.verbose = 0 # to remove the output of the solution process
    status = m.optimize()
    print(f"Number of necessary trucks: {m.objective_value}")
    for i in I:
        if y[i].x >= 0.99:
            pallets = [j+1 for j in J if x[i][j].x >= 0.99]
            print(f"pallets inside truck {i}: {pallets}")

solve_packing(truck_capacity, pallet_weights, 4)      